# UniAdvisor SLM training (Kaggle)
LoRA fine-tuning of `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` with typed heads (bool / choice / ordinal), temperature calibration and evaluation.

Settings: **GPU on, Internet on**. Input: the `uniadvisor-bundle` dataset (see kaggle/README.md).

In [ ]:
import glob, os, shutil, subprocess, zipfile
src = glob.glob('/kaggle/input/**/uniadvisor_kaggle_bundle.zip', recursive=True)
src_dir = glob.glob('/kaggle/input/**/uniadvisor/pyproject.toml', recursive=True)
if src:
    zipfile.ZipFile(src[0]).extractall('/kaggle/working')
elif src_dir:  # Kaggle unzips uploaded archives automatically
    shutil.copytree(os.path.dirname(src_dir[0]), '/kaggle/working/uniadvisor', dirs_exist_ok=True)
os.chdir('/kaggle/working/uniadvisor')
print(os.listdir('.'), os.listdir('data/slm'))

In [ ]:
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m uniadvisor.slm.train --data data/slm --out /kaggle/working/slm --epochs 3 --bs 64 --lr 5e-4 --lora-r 16

In [ ]:
import json
m = json.load(open('/kaggle/working/slm/metrics.json'))
print(json.dumps(m['test'], indent=1, ensure_ascii=False))
print(json.dumps(m['temperatures'], indent=1), json.dumps(m['thresholds'], indent=1))

In [ ]:
shutil.make_archive('/kaggle/working/slm_model', 'zip', '/kaggle/working/slm')
print('download /kaggle/working/slm_model.zip and unzip into models/slm/')